# Basic Operations

Using the F1 source data files in the folder "f1-sourcefiles/" create DataFrames.

This notebook covers:
- `show()`, `printSchema()`, `describe()`
- `select()`, `filter()` / `where()`, `limit()`
- `withColumn()`, `drop()`, `withColumnRenamed(old_name, new_name)`



In [1]:
# Initalise a spark session
import os
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Fix JAVA_HOME to your actual Java 21 path
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--packages io.delta:delta-spark_2.12:3.2.0 pyspark-shell"

# Build Spark session with Delta Lake support
builder = SparkSession.builder \
    .appName("DeltaLakeExample") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

26/03/25 23:20:17 WARN Utils: Your hostname, DESKTOP-OQT8U26 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/25 23:20:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/robyip/projects/pyspark-deltalake/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/robyip/.ivy2/cache
The jars for the packages stored in: /home/robyip/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-dd9a709d-cf9a-4954-93f4-29644504995b;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 156ms :: artifacts dl 6ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0 

In [2]:
# Load status.csv file
import time

start = time.time()

df = spark.read.csv("f1-sourcefiles/status.csv", header=True, inferSchema=True)

df.show()

end = time.time()

print(f"Time takes: {end - start:.2f} seconds")


26/03/25 23:20:34 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


+--------+------------+
|statusId|      status|
+--------+------------+
|       1|    Finished|
|       2|Disqualified|
|       3|    Accident|
|       4|   Collision|
|       5|      Engine|
|       6|     Gearbox|
|       7|Transmission|
|       8|      Clutch|
|       9|  Hydraulics|
|      10|  Electrical|
|      11|      +1 Lap|
|      12|     +2 Laps|
|      13|     +3 Laps|
|      14|     +4 Laps|
|      15|     +5 Laps|
|      16|     +6 Laps|
|      17|     +7 Laps|
|      18|     +8 Laps|
|      19|     +9 Laps|
|      20|    Spun off|
+--------+------------+
only showing top 20 rows

Time takes: 3.34 seconds


In [8]:
# Use commands after loading into a Dataframe
df.show()         # Preview data
df.printSchema()  # check column types
print("Row count: ", df.count())       # number of rows
print("Columnns: ", df.columns)       # list of column names

+--------+------------+
|statusId|      status|
+--------+------------+
|       1|    Finished|
|       2|Disqualified|
|       3|    Accident|
|       4|   Collision|
|       5|      Engine|
|       6|     Gearbox|
|       7|Transmission|
|       8|      Clutch|
|       9|  Hydraulics|
|      10|  Electrical|
|      11|      +1 Lap|
|      12|     +2 Laps|
|      13|     +3 Laps|
|      14|     +4 Laps|
|      15|     +5 Laps|
|      16|     +6 Laps|
|      17|     +7 Laps|
|      18|     +8 Laps|
|      19|     +9 Laps|
|      20|    Spun off|
+--------+------------+
only showing top 20 rows

root
 |-- statusId: integer (nullable = true)
 |-- status: string (nullable = true)

Row count:  139
Columnns:  ['statusId', 'status']


In [20]:
%%time
# Load status.csv file using a schema with %%time

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# start = time.time()

schema = StructType([
    StructField("statusId", IntegerType(), True),
    StructField("status", StringType(), True)
])

df = spark.read.csv("f1-sourcefiles/status.csv", header=True, schema=schema)

df.show()

# end = time.time()

# print(f"Time takes: {end - start:.2f} seconds")

df.printSchema()  # check column types
print("Row count: ", df.count())       # number of rows
print("Columnns: ", df.columns)       # list of column names

+--------+------------+
|statusId|      status|
+--------+------------+
|       1|    Finished|
|       2|Disqualified|
|       3|    Accident|
|       4|   Collision|
|       5|      Engine|
|       6|     Gearbox|
|       7|Transmission|
|       8|      Clutch|
|       9|  Hydraulics|
|      10|  Electrical|
|      11|      +1 Lap|
|      12|     +2 Laps|
|      13|     +3 Laps|
|      14|     +4 Laps|
|      15|     +5 Laps|
|      16|     +6 Laps|
|      17|     +7 Laps|
|      18|     +8 Laps|
|      19|     +9 Laps|
|      20|    Spun off|
+--------+------------+
only showing top 20 rows

root
 |-- statusId: integer (nullable = true)
 |-- status: string (nullable = true)

Row count:  139
Columnns:  ['statusId', 'status']
CPU times: user 7.7 ms, sys: 0 ns, total: 7.7 ms
Wall time: 137 ms


In [30]:
# select() - basic DataFrame operation

from pyspark.sql.functions import col

##  This is like a SELECT statement in SQL 

df.select("statusId").show()

##  This select all columns

df.select("*").show()

## Columns can be selected using a function called col() and alias() is used to rename a column name

df.select(col("statusId"), (col("statusId") + 10).alias("statusid-plus-10")).show()

## Using col() we can also use cast() to change the type

df.select(col("statusId"), (col("statusId") + 10).cast("double").alias("statusid-plus-10-cast-as-double")).show()


## Rule of thumb: If you just need to pick a column as-is, a plain string is fine. The moment you need to transform, rename, or calculate — use col().


+--------+
|statusId|
+--------+
|       1|
|       2|
|       3|
|       4|
|       5|
|       6|
|       7|
|       8|
|       9|
|      10|
|      11|
|      12|
|      13|
|      14|
|      15|
|      16|
|      17|
|      18|
|      19|
|      20|
+--------+
only showing top 20 rows

+--------+------------+
|statusId|      status|
+--------+------------+
|       1|    Finished|
|       2|Disqualified|
|       3|    Accident|
|       4|   Collision|
|       5|      Engine|
|       6|     Gearbox|
|       7|Transmission|
|       8|      Clutch|
|       9|  Hydraulics|
|      10|  Electrical|
|      11|      +1 Lap|
|      12|     +2 Laps|
|      13|     +3 Laps|
|      14|     +4 Laps|
|      15|     +5 Laps|
|      16|     +6 Laps|
|      17|     +7 Laps|
|      18|     +8 Laps|
|      19|     +9 Laps|
|      20|    Spun off|
+--------+------------+
only showing top 20 rows

+--------+----------------+
|statusId|statusid-plus-10|
+--------+----------------+
|       1|              

In [4]:
# filter() - basic operations on dataframes using col()

from pyspark.sql.functions import col

df.filter(col("statusId") < 13).show()


+--------+------------+
|statusId|      status|
+--------+------------+
|       1|    Finished|
|       2|Disqualified|
|       3|    Accident|
|       4|   Collision|
|       5|      Engine|
|       6|     Gearbox|
|       7|Transmission|
|       8|      Clutch|
|       9|  Hydraulics|
|      10|  Electrical|
|      11|      +1 Lap|
|      12|     +2 Laps|
+--------+------------+



In [6]:
# filter() - basic operations on dataframes using SQL string expression

df.filter("statusId < 7").show()

+--------+------------+
|statusId|      status|
+--------+------------+
|       1|    Finished|
|       2|Disqualified|
|       3|    Accident|
|       4|   Collision|
|       5|      Engine|
|       6|     Gearbox|
+--------+------------+



In [7]:
# filter() - basic operations on dataframes using SQL string expression and exact match

df.filter("status == 'Engine'").show()



+--------+------+
|statusId|status|
+--------+------+
|       5|Engine|
+--------+------+



In [8]:
# filter() - basic operations on dataframes using col() and exact match

df.filter(col("status") == "Engine").show()



+--------+------+
|statusId|status|
+--------+------+
|       5|Engine|
+--------+------+



In [3]:
# limit() basic operations on dataframes

# Usage:  limit() restricts the number of rows returned from a DataFrame — similar to LIMIT in SQL

# df.limit(n)

df.limit(5).show()

+--------+------------+
|statusId|      status|
+--------+------------+
|       1|    Finished|
|       2|Disqualified|
|       3|    Accident|
|       4|   Collision|
|       5|      Engine|
+--------+------------+



In [6]:
# limit() chained with other transformation

df.filter("statusId < 99") \
    .select("statusId", "status") \
    .orderBy("statusId", ascending=False) \
    .limit(20) \
    .show()

+--------+------------------+
|statusId|            status|
+--------+------------------+
|      98|         Injection|
|      97|Did not prequalify|
|      96|          Excluded|
|      95|         Fuel leak|
|      94|          Oil pump|
|      93|       Safety belt|
|      92|       Underweight|
|      91|        Alternator|
|      90|     Not restarted|
|      89|   Safety concerns|
|      88|          +10 Laps|
|      87|        Crankshaft|
|      86|         Halfshaft|
|      85|           Stalled|
|      84|           Battery|
|      83|           Chassis|
|      82|            Injury|
|      81|   Did not qualify|
|      80|          Ignition|
|      79|        Drivetrain|
+--------+------------------+



In [7]:
# limit() in SQL

# This take the populated Dataframe and creates temporary view
df.createOrReplaceTempView("status")

# This takes that temporary view and uses it in a SQL SELECT 
spark.sql("SELECT * FROM status LIMIT 5").show()



+--------+------------+
|statusId|      status|
+--------+------------+
|       1|    Finished|
|       2|Disqualified|
|       3|    Accident|
|       4|   Collision|
|       5|      Engine|
+--------+------------+



In [ ]:
# withColumn() is used to add a new column or replace an existing column in a DataFrame. 
# It always returns a new DataFrame (DataFrames are immutable in Spark).

# Basic syntax: df.withColumn("column_name", column_expression)

# 1st argument — name of the new or existing column (string)
# 2nd argument — the column expression/transformation to apply



In [9]:
# Adding a new column using .withColumn()

from pyspark.sql.functions import col

df.withColumn("statusId_doubled", col("statusId") * 2).show()

+--------+------------+----------------+
|statusId|      status|statusId_doubled|
+--------+------------+----------------+
|       1|    Finished|               2|
|       2|Disqualified|               4|
|       3|    Accident|               6|
|       4|   Collision|               8|
|       5|      Engine|              10|
|       6|     Gearbox|              12|
|       7|Transmission|              14|
|       8|      Clutch|              16|
|       9|  Hydraulics|              18|
|      10|  Electrical|              20|
|      11|      +1 Lap|              22|
|      12|     +2 Laps|              24|
|      13|     +3 Laps|              26|
|      14|     +4 Laps|              28|
|      15|     +5 Laps|              30|
|      16|     +6 Laps|              32|
|      17|     +7 Laps|              34|
|      18|     +8 Laps|              36|
|      19|     +9 Laps|              38|
|      20|    Spun off|              40|
+--------+------------+----------------+
only showing top

In [10]:
# Replacing or updating an existing column using .withColumn()

from pyspark.sql.functions import col

df.withColumn("statusId", col("statusId") * 100).show()

+--------+------------+
|statusId|      status|
+--------+------------+
|     100|    Finished|
|     200|Disqualified|
|     300|    Accident|
|     400|   Collision|
|     500|      Engine|
|     600|     Gearbox|
|     700|Transmission|
|     800|      Clutch|
|     900|  Hydraulics|
|    1000|  Electrical|
|    1100|      +1 Lap|
|    1200|     +2 Laps|
|    1300|     +3 Laps|
|    1400|     +4 Laps|
|    1500|     +5 Laps|
|    1600|     +6 Laps|
|    1700|     +7 Laps|
|    1800|     +8 Laps|
|    1900|     +9 Laps|
|    2000|    Spun off|
+--------+------------+
only showing top 20 rows



In [11]:
#  Adding a constant/literal value using .withColumn()

from pyspark.sql.functions import lit

df.withColumn("country", lit("UK")).show()

+--------+------------+-------+
|statusId|      status|country|
+--------+------------+-------+
|       1|    Finished|     UK|
|       2|Disqualified|     UK|
|       3|    Accident|     UK|
|       4|   Collision|     UK|
|       5|      Engine|     UK|
|       6|     Gearbox|     UK|
|       7|Transmission|     UK|
|       8|      Clutch|     UK|
|       9|  Hydraulics|     UK|
|      10|  Electrical|     UK|
|      11|      +1 Lap|     UK|
|      12|     +2 Laps|     UK|
|      13|     +3 Laps|     UK|
|      14|     +4 Laps|     UK|
|      15|     +5 Laps|     UK|
|      16|     +6 Laps|     UK|
|      17|     +7 Laps|     UK|
|      18|     +8 Laps|     UK|
|      19|     +9 Laps|     UK|
|      20|    Spun off|     UK|
+--------+------------+-------+
only showing top 20 rows



In [10]:
# Using conditionals (when/otherwise) - using .withColumn()

from pyspark.sql import functions as F

df.withColumn("odds or even", \
                F.when(F.col("statusId") % 2 == 1, "Odds") \
                .when(F.col("statusId") % 2 == 0, "Evens") \
                .otherwise("Neither")).show()

              

+--------+------------+------------+
|statusId|      status|odds or even|
+--------+------------+------------+
|       1|    Finished|        Odds|
|       2|Disqualified|       Evens|
|       3|    Accident|        Odds|
|       4|   Collision|       Evens|
|       5|      Engine|        Odds|
|       6|     Gearbox|       Evens|
|       7|Transmission|        Odds|
|       8|      Clutch|       Evens|
|       9|  Hydraulics|        Odds|
|      10|  Electrical|       Evens|
|      11|      +1 Lap|        Odds|
|      12|     +2 Laps|       Evens|
|      13|     +3 Laps|        Odds|
|      14|     +4 Laps|       Evens|
|      15|     +5 Laps|        Odds|
|      16|     +6 Laps|       Evens|
|      17|     +7 Laps|        Odds|
|      18|     +8 Laps|       Evens|
|      19|     +9 Laps|        Odds|
|      20|    Spun off|       Evens|
+--------+------------+------------+
only showing top 20 rows



## The key distinction:

F.when() — when is a module-level function inside pyspark.sql.functions. You call it via F because that's the module alias. It starts the chain and returns a Column.

.when() (no F.) — when is also a method on the Column class. Once the first F.when() gives you back a Column object, all the subsequent .when() and .otherwise() calls are just chained methods on that object — same as how you'd chain .strip().lower().split() on a Python string.

In [11]:
# .drop() - basic operations

df.drop("status").show()


+--------+
|statusId|
+--------+
|       1|
|       2|
|       3|
|       4|
|       5|
|       6|
|       7|
|       8|
|       9|
|      10|
|      11|
|      12|
|      13|
|      14|
|      15|
|      16|
|      17|
|      18|
|      19|
|      20|
+--------+
only showing top 20 rows



## Most common real-world uses:

- Cleaning up after a join — joins often bring in duplicate or helper columns you no longer need
- Removing staging/intermediate columns added earlier in a pipeline (e.g. a temp_score you used in a when() expression)
- Dropping sensitive fields before writing data out

One gotcha — if two columns share the same name (can happen after a join), passing a string to drop() will remove both. To drop just one, you need to reference it by its full qualified name or restructure the join first.

In [12]:
# Renaming columns using - withColumnRenamed(old_name, new_name)

df.withColumnRenamed("status", "TheStatus").show()

+--------+------------+
|statusId|   TheStatus|
+--------+------------+
|       1|    Finished|
|       2|Disqualified|
|       3|    Accident|
|       4|   Collision|
|       5|      Engine|
|       6|     Gearbox|
|       7|Transmission|
|       8|      Clutch|
|       9|  Hydraulics|
|      10|  Electrical|
|      11|      +1 Lap|
|      12|     +2 Laps|
|      13|     +3 Laps|
|      14|     +4 Laps|
|      15|     +5 Laps|
|      16|     +6 Laps|
|      17|     +7 Laps|
|      18|     +8 Laps|
|      19|     +9 Laps|
|      20|    Spun off|
+--------+------------+
only showing top 20 rows

